## 1. Set up your environment

This step ensures all the necessary tools are available to build the RAG system. Each library serves a specific role: Langchain handles the orchestration of components, transformers provide pre-trained models, sentence-transformers generate embeddings, datasets load sample data, and FAISS enables fast similarity searches.

In [1]:
# Install required libraries
!pip install -q langchain
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q datasets
!pip install -q faiss-cpu
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 2. Load the dataset

To provide the system with information to retrieve from, you’ll load a real-world dataset. HuggingFaceDatasetLoader simplifies the process of accessing Hugging Face datasets and formatting them into documents that Langchain can process.

In [3]:
# Install datasets library (if not already installed or to ensure latest version)
!pip install -Uq datasets

# Import HuggingFaceDatasetLoader
from langchain_community.document_loaders import HuggingFaceDatasetLoader

# Specify dataset name and content column
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

# Create HuggingFaceDatasetLoader instance and load data as documents
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

# Optional: Print the first 2 entries to verify loading
print(data[:2])

[Document(metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}, page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia\'s domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."'), Document(metadata={'instruction': 'Which is a species of fish? Tope or Rope', 'response': 'Tope', 'category': 'classification'}, page_content='""')]


## 3. Diviser les documents (Split the documents)

Les modèles linguistiques ont une limite à la quantité de texte qu'ils peuvent traiter à la fois. La division de documents volumineux en morceaux plus petits et superposés garantit qu'aucun contexte important n'est perdu et que chaque morceau de texte a une taille gérable pour l'intégration et la récupération.

In [5]:
# Import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a RecursiveCharacterTextSplitter instance
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

# Split the loaded documents
docs = text_splitter.split_documents(data)

# Optional: Print the first document chunk
print(docs[0])

page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


## 4. Intégrer le texte (Embed the text)

Le texte doit être converti en représentations numériques (intégrations) afin que des morceaux de texte similaires puissent être trouvés efficacement. L’utilisation d’un modèle de transformateur de phrases crée des intégrations qui capturent le sens sémantique, permettant une récupération efficace ultérieurement.

In [6]:
# Import HuggingFaceEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Define model path, model configurations, and encoding options
modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {'device':'cpu'}
encode_kwargs = {'normalize_embeddings': False}

# Initialize HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
  model_name=modelPath,
  model_kwargs=model_kwargs,
  encode_kwargs=encode_kwargs
)

# (Optional) Creating test embedding:
text = "This is a test document."
query_result = embeddings.embed_query(text)
print(query_result[:3])

/tmp/ipykernel_2297/3814392946.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[-0.038338519632816315, 0.1234646886587143, -0.028642984107136726]


## 5. Créer un magasin vectoriel (Create a vector store)

Un magasin vectoriel comme FAISS indexe les intégrations, permettant des recherches de similarité rapides et évolutives. C'est ainsi que le système trouve rapidement les morceaux de texte pertinents lorsqu'une requête est effectuée.

In [13]:
# Import FAISS
from langchain_community.vectorstores import FAISS

# Create a FAISS vector store from document chunks and embeddings
db = FAISS.from_documents(docs, embeddings)

print("FAISS vector store created successfully.")

KeyboardInterrupt: 

## 6. Prepare the LLM model

The Language Model is responsible for generating answers based on retrieved documents. Loading a pre-trained model and wrapping it in a Langchain pipeline makes it easy to integrate with the retrieval system.

In [9]:
# Import necessary classes from transformers and langchain
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

# Load the tokenizer and question-answering model
tokenizer = AutoTokenizer.from_pretrained("Intel/dynamic_tinybert")
model = AutoModelForQuestionAnswering.from_pretrained("Intel/dynamic_tinybert")

# Create a question-answering pipeline
model_name = "Intel/dynamic_tinybert"
qa_pipeline = pipeline(
  "question-answering",
  model=model_name,
  tokenizer=tokenizer,
  return_tensors='pt'
)

# Create a Langchain pipeline wrapper
llm = HuggingFacePipeline(
  pipeline=qa_pipeline,
  model_kwargs={"temperature": 0.7, "max_length": 512},
)

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: Intel/dynamic_tinybert
Key              | Status     |  | 
-----------------+------------+--+-
fit_dense.bias   | UNEXPECTED |  | 
fit_dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

KeyError: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

## 7. Build the Retrieval QA Chain

The Retrieval QA Chain connects the retriever (which finds relevant documents) with the LLM (which generates answers). This chain enables the full RAG process, where the system retrieves helpful context and then answers the user’s query based on that context.

In [10]:
# Import RetrievalQA
from langchain.chains import RetrievalQA

# Create a retriever from your FAISS database
retriever = db.as_retriever(search_kwargs={"k": 4}) # Optional: You can adjust k for number of documents retrieved

# Build the RetrievalQA chain
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="refine", retriever=retriever, return_source_documents=False)

print("Retrieval QA Chain built successfully.")

ModuleNotFoundError: No module named 'langchain.chains'

## 8. Test your RAG system

Running a test query allows you to verify that all components are working together. This step ensures that documents are retrieved correctly and that the model generates meaningful answers based on the retrieved context.

In [11]:
# Define your question
question = "What is cheesemaking?"

# Run the QA chain and print the result
result = qa.run({"query": question})
print(result)

NameError: name 'qa' is not defined